<a href="https://colab.research.google.com/github/CavalierTomWard/SMS-CL_automations/blob/main/SMSB%26R%2C_SMS_and_calls_merge_per_contact_statistics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
import pandas as pd
import numpy as np
import xml.etree.ElementTree as ET
import re
import os

# 1. Mount Google Drive
drive.mount('/content/drive')
folder_path = '/content/drive/MyDrive/SMSB&RP'
sms_csv_path = os.path.join(folder_path, 'sms_text_only_2026-10-28.csv')
calls_xml_path = os.path.join(folder_path, 'calls-2026-10-28_00-56-16.xml')
output_csv_path = os.path.join(folder_path, 'master_contact_stats_2026-10-28.csv')

# 2. Phone Number Normalization (Australian +61 standard)
def normalize_phone(num):
    if pd.isna(num):
        return 'Unknown'
    s = str(num).strip()
    # Keep alphanumeric sender IDs intact (e.g. "AusPost", " ANZ ")
    if any(c.isalpha() for c in s):
        return s
    # Strip spaces, dashes, brackets
    cleaned = re.sub(r'[^\d+]', '', s)
    if not cleaned:
        return s
    if cleaned.startswith('0011'):
        cleaned = '+' + cleaned[4:]
    elif cleaned.startswith('0') and len(cleaned) == 10:
        # 04xx xxx xxx or 02 xxxx xxxx -> +614... / +612...
        cleaned = '+61' + cleaned[1:]
    elif cleaned.startswith('61') and len(cleaned) == 11:
        cleaned = '+' + cleaned
    elif len(cleaned) == 9 and cleaned.startswith('4'):
        cleaned = '+61' + cleaned
    return cleaned

# 3. Load & Prepare SMS Data
print("Loading SMS CSV...")
df_sms = pd.read_csv(sms_csv_path, dtype=str)
df_sms['norm_number'] = df_sms['address'].apply(normalize_phone)
df_sms['dt'] = pd.to_datetime(df_sms['readable_date'], errors='coerce')
df_sms['body'] = df_sms['body'].fillna('')
df_sms['word_count'] = df_sms['body'].apply(lambda x: len(x.split()))
df_sms = df_sms.dropna(subset=['dt']).sort_values(['norm_number', 'dt'])

# 4. Load & Prepare Call Logs XML
print("Loading Calls XML...")
calls_list = []
for event, elem in ET.iterparse(calls_xml_path, events=('end',)):
    if elem.tag == 'call':
        calls_list.append({
            'number': elem.attrib.get('number', ''),
            'contact_name': elem.attrib.get('contact_name', '(Unknown)'),
            'readable_date': elem.attrib.get('readable_date', ''),
            'duration': int(elem.attrib.get('duration', 0)),
            'call_type': elem.attrib.get('type', '') # 1=Incoming, 2=Outgoing, 3=Missed, 5=Rejected
        })
        elem.clear()

df_calls = pd.DataFrame(calls_list)
df_calls['norm_number'] = df_calls['number'].apply(normalize_phone)
df_calls['dt'] = pd.to_datetime(df_calls['readable_date'], errors='coerce')
df_calls = df_calls.dropna(subset=['dt']).sort_values(['norm_number', 'dt'])

# 5. Build Contact Name Lookup (prefer real contact names over '(Unknown)')
name_map = {}
for df in [df_sms, df_calls]:
    valid_names = df[~df['contact_name'].isin(['(Unknown)', '', 'null', None])].dropna(subset=['contact_name'])
    for num, name in zip(valid_names['norm_number'], valid_names['contact_name']):
        name_map[num] = name

# 6. Compute Per-Contact SMS Metrics (Volume, Initiations, Response Times)
print("Calculating SMS conversation dynamics...")
sms_stats = {}

for num, group in df_sms.groupby('norm_number', sort=False):
    types = group['type'].values
    dts = group['dt'].values
    words = group['word_count'].values

    sent_mask = (types == 'Sent')
    rec_mask = (types == 'Received')

    sms_sent = int(np.sum(sent_mask))
    sms_rec = int(np.sum(rec_mask))
    words_sent = int(np.sum(words[sent_mask]))
    words_rec = int(np.sum(words[rec_mask]))

    # Time diffs in seconds between consecutive messages
    if len(dts) > 1:
        diffs_sec = (dts[1:] - dts[:-1]) / np.timedelta64(1, 's')
    else:
        diffs_sec = np.array([])

    # Conversation Initiations (First message OR message after a 6+ hour gap)
    init_you = 0
    init_them = 0
    if len(types) > 0:
        if types[0] == 'Sent':
            init_you += 1
        else:
            init_them += 1

        gap_indices = np.where(diffs_sec >= 6 * 3600)[0] + 1
        for idx in gap_indices:
            if types[idx] == 'Sent':
                init_you += 1
            else:
                init_them += 1

    # Response Times (Direction switches within 12 hours, ignoring 0s multi-part splits)
    prev_types = types[:-1]
    curr_types = types[1:]
    valid_reply_window = (diffs_sec > 0) & (diffs_sec <= 12 * 3600)

    your_replies = diffs_sec[(prev_types == 'Received') & (curr_types == 'Sent') & valid_reply_window]
    their_replies = diffs_sec[(prev_types == 'Sent') & (curr_types == 'Received') & valid_reply_window]

    sms_stats[num] = {
        'sms_total': sms_sent + sms_rec,
        'sms_sent': sms_sent,
        'sms_received': sms_rec,
        'words_sent': words_sent,
        'words_received': words_rec,
        'avg_words_sent': round(words_sent / sms_sent, 1) if sms_sent else 0.0,
        'avg_words_received': round(words_rec / sms_rec, 1) if sms_rec else 0.0,
        'initiations_you': init_you,
        'initiations_them': init_them,
        'you_initiate_pct': round(init_you / (init_you + init_them) * 100, 1) if (init_you + init_them) else 0.0,
        'your_median_reply_mins': round(float(np.median(your_replies)) / 60.0, 1) if len(your_replies) else None,
        'their_median_reply_mins': round(float(np.median(their_replies)) / 60.0, 1) if len(their_replies) else None,
        'first_sms': pd.Timestamp(dts[0]),
        'last_sms': pd.Timestamp(dts[-1]),
        'sms_active_days': group['dt'].dt.date.nunique()
    }

# 7. Compute Per-Contact Call Metrics
print("Calculating Call metrics...")
call_stats = {}

for num, group in df_calls.groupby('norm_number', sort=False):
    ctypes = group['call_type'].values
    durs = group['duration'].values
    dts = group['dt'].values

    inc_mask = (ctypes == '1')
    out_mask = (ctypes == '2')
    miss_mask = np.isin(ctypes, ['3', '5']) # Missed or Rejected

    calls_in = int(np.sum(inc_mask))
    calls_out = int(np.sum(out_mask))
    calls_missed = int(np.sum(miss_mask))

    dur_in_sec = int(np.sum(durs[inc_mask]))
    dur_out_sec = int(np.sum(durs[out_mask]))
    total_dur_sec = dur_in_sec + dur_out_sec
    connected_calls = int(np.sum(durs > 0))

    call_stats[num] = {
        'calls_total': len(group),
        'calls_incoming': calls_in,
        'calls_outgoing': calls_out,
        'calls_missed': calls_missed,
        'total_call_mins': round(total_dur_sec / 60.0, 1),
        'incoming_call_mins': round(dur_in_sec / 60.0, 1),
        'outgoing_call_mins': round(dur_out_sec / 60.0, 1),
        'avg_connected_call_mins': round((total_dur_sec / connected_calls) / 60.0, 1) if connected_calls else 0.0,
        'longest_call_mins': round(int(np.max(durs)) / 60.0, 1) if len(durs) else 0.0,
        'first_call': pd.Timestamp(dts[0]),
        'last_call': pd.Timestamp(dts[-1]),
        'call_active_days': group['dt'].dt.date.nunique()
    }

# 8. Merge SMS & Call Stats into Master Table
print("Merging and exporting master CSV...")
all_numbers = set(sms_stats.keys()).union(set(call_stats.keys()))
rows = []

for num in all_numbers:
    s = sms_stats.get(num, {})
    c = call_stats.get(num, {})

    first_dates = [d for d in [s.get('first_sms'), c.get('first_call')] if d is not None]
    last_dates = [d for d in [s.get('last_sms'), c.get('last_call')] if d is not None]

    sms_tot = s.get('sms_total', 0)
    calls_tot = c.get('calls_total', 0)

    rows.append({
        'phone_number': num,
        'contact_name': name_map.get(num, '(Unknown)'),
        'total_interactions': sms_tot + calls_tot,
        # SMS Volume
        'sms_total': sms_tot,
        'sms_sent': s.get('sms_sent', 0),
        'sms_received': s.get('sms_received', 0),
        'words_sent': s.get('words_sent', 0),
        'words_received': s.get('words_received', 0),
        'avg_words_sent': s.get('avg_words_sent', 0.0),
        'avg_words_received': s.get('avg_words_received', 0.0),
        # Initiation & Response Times
        'initiations_you': s.get('initiations_you', 0),
        'initiations_them': s.get('initiations_them', 0),
        'you_initiate_pct': s.get('you_initiate_pct', 0.0),
        'your_median_reply_mins': s.get('your_median_reply_mins', ''),
        'their_median_reply_mins': s.get('their_median_reply_mins', ''),
        # Call Metrics
        'calls_total': calls_tot,
        'calls_incoming': c.get('calls_incoming', 0),
        'calls_outgoing': c.get('calls_outgoing', 0),
        'calls_missed': c.get('calls_missed', 0),
        'total_call_mins': c.get('total_call_mins', 0.0),
        'avg_connected_call_mins': c.get('avg_connected_call_mins', 0.0),
        'longest_call_mins': c.get('longest_call_mins', 0.0),
        # Activity Timeline
        'sms_active_days': s.get('sms_active_days', 0),
        'call_active_days': c.get('call_active_days', 0),
        'first_interaction': min(first_dates).strftime('%Y-%m-%d %H:%M') if first_dates else '',
        'last_interaction': max(last_dates).strftime('%Y-%m-%d %H:%M') if last_dates else ''
    })

df_master = pd.DataFrame(rows)
df_master = df_master.sort_values(by=['total_interactions', 'total_call_mins'], ascending=[False, False])
df_master.to_csv(output_csv_path, index=False, encoding='utf-8')

print(f"Done! Exported {len(df_master)} unique contacts to {output_csv_path}")

Mounted at /content/drive
Loading SMS CSV...
Loading Calls XML...
Calculating SMS conversation dynamics...
Calculating Call metrics...
Merging and exporting master CSV...
Done! Exported 1290 unique contacts to /content/drive/MyDrive/SMSB&RP/master_contact_stats_2026-10-28.csv
